In [44]:
import os, glob
import pandas as pd
import numpy as np

Building the manifest(to list what all faults have how many csv files)

In [45]:
DATA_ROOT = '../data/raw/mafaulda'
records = []
rows = []

for i in glob.glob(f"{DATA_ROOT}/normal/*.csv"):
    rows.append({"path": i, "label" : "normal", "location" : None})

for i in["underhang", "overhang"]:
    for j in glob.glob(f"{DATA_ROOT}/{i}/*/*/*.csv"):
        parts = j.split(os.sep)
        rows.append({"path" : j , "label" : parts[-3], "location" : i})

mani = pd.DataFrame(rows)
mani.groupby(["label", "location"], dropna = False ).size()


label       location 
ball_fault  overhang     137
            underhang    186
cage_fault  overhang     188
            underhang    188
normal      NaN           49
outer_race  overhang     188
            underhang    184
dtype: int64

Next cell is basically just oicking out signals like which one to use 


In [46]:
COLS = {"underhang" : [1,2,3], "overhang" : [4,5,6]}

def get_signal(path,loaction):
    df = pd.read_csv(path, header = None)
    return df.iloc[:, COLS[loaction][1]].values

Next cell is segmenting data

In [47]:
Window = 4096
Step = 2048
def segment(sig, window=Window, step = Step):
    n= (len(sig) - window) // step + 1
    return np.array([sig[i*step : i*step+window] for i in range(n)])


In [30]:
X, y, loc_tags = [], [], []
for _, row in mani.iterrows():
    if row.label == "normal":
        for i in ["underhang", "overhang"]:
            w = segment(get_signal(row.path,i))
            X.append(w)
            y += ["normal"]*len(w)
            loc_tags += [i] * len(w)
    else :
        w = segment(get_signal(row.path,row.location))
        X.append(w)
        y += [row.label]*len(w)
        loc_tags += [row.location]*len(w)

X = np.vstack(X)
y = np.array(y)
loc_tags = np.array(loc_tags)
X.shape, y.shape

((141449, 4096), (141449,))

In [34]:
from scipy.stats import kurtosis
from scipy.fft import fft, fftfreq

def extract(w, fs = 50000):
    rms = np.sqrt(np.mean(w**2))
    kurt = kurtosis(w)
    crest = np.max(np.abs(w)) / rms

    n=len(w)
    freq = fftfreq(n, 1/fs)[:n//2]
    mag = np.abs(fft(w))[:n//2]*2/n
    edges = np.linspace(0, fs/2, 6)
    bands = {}

    for i in range(5):
        mask = (freq >= edges[i]) & (freq<edges[i+1])
        bands[f"band_{i}"] = np.sum(mag[mask]**2)
    return{"rms" : rms, "kurtosis" : kurt, "crest" : crest, **bands}

In [35]:
feat_df = pd.DataFrame([extract(w) for w in X])
feat_df["label"] = y
feat_df["location"] = loc_tags
feat_df.groupby(["label","location"]).size()

label       location 
ball_fault  overhang     16577
            underhang    22506
cage_fault  overhang     22748
            underhang    22748
normal      overhang      5929
            underhang     5929
outer_race  overhang     22748
            underhang    22264
dtype: int64

In [48]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

feat = [ c for c in feat_df.columns if c not in ("label", "location")]

Xtr, Xte, ytr, yte = train_test_split(
    feat_df[feat], feat_df["label"], test_size = 0.25,
    stratify = feat_df["label"], random_state = 0
)

clf = RandomForestClassifier(n_estimators = 200, random_state = 0)
clf.fit(Xtr, ytr)
print("combined random split : ", clf.score(Xte, yte))


combined random split :  0.9654158301049119


In [54]:
train_idx = feat_df[feat_df.location == "underhang"].index.tolist()
test_idx = feat_df[feat_df.location == "overhang"].index.tolist()

clf2 = RandomForestClassifier(n_estimators=200, random_state=0)
clf2.fit(feat_df.loc[train_idx, feat], feat_df.loc[train_idx, "label"])
print("trained underhang, tested overhang:", clf2.score(feat_df.loc[test_idx, feat], feat_df.loc[test_idx, "label"]))

trained underhang, tested overhang: 0.31991706126290403


In [55]:
from sklearn.metrics import confusion_matrix

preds = clf2.predict(feat_df.loc[test_idx, feat])
actual = feat_df.loc[test_idx, "label"]

labels = sorted(actual.unique())
cm = confusion_matrix(actual, preds, labels=labels)
pd.DataFrame(cm, index=labels, columns=labels)

,ball_fault,cage_fault,normal,outer_race
ball_fault,2495,225,0,13857
cage_fault,352,16285,311,5800
normal,0,5903,18,8
outer_race,0,19646,145,2957


In [63]:
def extract_norm(w, fs = 50000):
    rms = np.sqrt(np.mean(w**2))
    wn = w/rms
    kurt = kurtosis(wn)
    crest = np.max(np.abs(wn))

    n = len(wn)
    freq = fftfreq(n, 1/fs)[:n//2]
    mag = np.abs(fft(wn))[:n//2]*2/n
    edges = np.linspace(0,fs/2,6)
    bands = {}

    for i in range(5):
        mask = (freq >= edges[i]) & (freq < edges[i+1])
        bands[f"band_{i}"] = np.sum(mag[mask]**2)

    return {"kurtosis" : kurt, "crest" : crest, **bands}


In [64]:
feat_df_norm = pd.DataFrame([extract_norm(w) for w in X])
feat_df_norm["label"] = y
feat_df_norm["location"] = loc_tags
feat_df_norm.groupby(["label", "location"]).size()

label       location 
ball_fault  overhang     16577
            underhang    22506
cage_fault  overhang     22748
            underhang    22748
normal      overhang      5929
            underhang     5929
outer_race  overhang     22748
            underhang    22264
dtype: int64

In [65]:
feat_n = [c for c in feat_df_norm.columns if c not in ("label", "location")]

clf3 = RandomForestClassifier(n_estimators=200, random_state=0)
clf3.fit(feat_df_norm.loc[train_idx, feat_n], feat_df_norm.loc[train_idx, "label"])
print("trained underhang, tested overhang:", clf3.score(feat_df_norm.loc[test_idx, feat_n], feat_df_norm.loc[test_idx, "label"]),"(normalized)")

trained underhang, tested overhang: 0.1749948530925561 (normalized)


In [66]:
preds_norm = clf3.predict(feat_df_norm.loc[test_idx, feat_n])
actual_norm = feat_df_norm.loc[test_idx, "label"]
labels = sorted(actual_norm.unique())
cm_norm = confusion_matrix(actual_norm, preds_norm, labels = labels)
pd.DataFrame(cm_norm, index = labels, columns = labels)

,ball_fault,cage_fault,normal,outer_race
ball_fault,971,0,0,15606
cage_fault,3560,4899,0,14289
normal,466,5068,0,395
outer_race,2563,14155,0,6030


In [69]:
def extract_hybrid(w, fs = 50000):
    rms = np.sqrt(np.mean(w**2))
    wn = w/rms

    kurt_raw = kurtosis(w)
    crest_raw = np.max(np.abs(w))/rms
    kurt_norm = kurtosis(wn)
    crest_norm = np.max(np.abs(wn))

    n=len(w)
    freq = fftfreq(n,1/fs)[:n//2]
    mag_raw = np.abs(fft(w))[:n//2]*2/n
    mag_norm = np.abs(fft(wn))[:n//2]*2/n
    edges = np.linspace(0, fs/2, 6)

    feats = {"rms" : rms, "kurtosis_raw" : kurt_raw, "crest_raw" : crest_raw,
             "kurtosis_norm": kurt_norm, "crest_norm" : crest_norm}
    
    for i in range(5):
        mask = (freq >= edges[i]) & (freq < edges[i+1])
        feats[f"band_raw_{i}"] = np.sum(mag_raw[mask]**2)
        feats[f"band_norm_{i}"] = np.sum(mag_norm[mask]**2)
    return feats

In [71]:
feat_df_hybrid = pd.DataFrame([extract_hybrid(w) for w in X])
feat_df_hybrid["label"] = y
feat_df_hybrid["location"] = loc_tags
feat_df_hybrid.groupby(["label", "location"]).size()

label       location 
ball_fault  overhang     16577
            underhang    22506
cage_fault  overhang     22748
            underhang    22748
normal      overhang      5929
            underhang     5929
outer_race  overhang     22748
            underhang    22264
dtype: int64

In [72]:
feat_n = [c for c in feat_df_hybrid.columns if c not in ("label", "location")]

clf3 = RandomForestClassifier(n_estimators=200, random_state=0)
clf3.fit(feat_df_hybrid.loc[train_idx, feat_n], feat_df_hybrid.loc[train_idx, "label"])
print("trained underhang, tested overhang:", clf3.score(feat_df_hybrid.loc[test_idx, feat_n], feat_df_hybrid.loc[test_idx, "label"]),"(hybrid)")

trained underhang, tested overhang: 0.21249375018381814 (hybrid)


In [74]:
preds_hybrid = clf3.predict(feat_df_hybrid.loc[test_idx, feat_n])
actual_hybrid = feat_df_hybrid.loc[test_idx, "label"]
labels = sorted(actual_hybrid.unique())
cm_hybrid = confusion_matrix(actual_hybrid, preds_hybrid, labels = labels)
pd.DataFrame(cm_hybrid, index = labels, columns = labels)

,ball_fault,cage_fault,normal,outer_race
ball_fault,537,8,0,16032
cage_fault,1100,3716,0,17932
normal,0,4880,0,1049
outer_race,1,12550,0,10197


In [ ]:
for i in range